# Decision Trees and Random Forest

This notebook provides a university-level introduction to Decision Trees (CART) and Random Forests, covering both theoretical foundations and practical implementations in Python.

Author: Michał Woźniak

## Technical Setup

This notebook is part of the course repository, which already ships a complete, locked environment (`pyproject.toml` + `uv.lock`). You do **not** need to install packages one by one.

```bash
# from the root of the repository
uv sync
uv run jupyter lab
```

A detailed, step-by-step installation guide (what `uv` is, how to install it on Windows / macOS / Linux, how to select the kernel in VS Code, troubleshooting) is available in [`SETUP.md`](../SETUP.md).

**Packages used in this notebook:** `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`.

**Data:** the German Credit dataset used in Section 6 ships with the repository (`data/german_credit.csv`). California Housing (`fetch_california_housing`) is downloaded on first use and then cached locally in `~/scikit_learn_data`, so the first run needs internet access.

# 1. Introduction to Decision Trees and Random Forest

## What is a Decision Tree?

A Decision Tree is a **non-parametric** supervised learning model used for both classification and regression. It learns a hierarchy of simple **if-else decision rules** from the training data — "if the debt-to-income ratio is above 0.43 and the client has had a delinquency in the last 12 months, then predict *default*".

### Key Characteristics:

1. **Non-parametric**: no assumption about the distribution of the data or the functional form of $f(X)$; the number of "parameters" (splits) grows with the data.

2. **Piecewise-constant approximation**: a tree partitions the feature space into axis-aligned rectangles (the leaves) $R_1, \dots, R_M$ and predicts a constant in each of them:
   $$\hat{f}(x) = \sum_{m=1}^{M} c_m \cdot \mathbb{1}(x \in R_m)$$
   where $c_m$ is the mean of the target in leaf $m$ (regression) or the vector of class proportions in leaf $m$ (classification).

3. **Greedy, recursive training**: finding the optimal tree is NP-hard, so every practical algorithm builds the tree top-down, choosing the locally best split at each node and never revisiting it.

4. **A family of algorithms**: ID3 (1979/1986), CART (1984), C4.5 (1993), C5.0 (late 1990s). scikit-learn implements an optimised version of **CART** (Classification and Regression Trees): binary splits, numerical features, both classification and regression. This is the algorithm we derive and implement below.

### How CART Works:

Let the data at node $m$ be $Q_m$ with $n_m$ observations. A candidate split $\theta = (j, t_m)$ consists of a feature $j$ and a threshold $t_m$, and sends observations to the left or to the right child:

$$Q_m^{left}(\theta) = \{(x, y) \mid x_j \leq t_m\}, \qquad Q_m^{right}(\theta) = Q_m \setminus Q_m^{left}(\theta)$$

The quality of a split is the **weighted impurity of the children**:

$$G(Q_m, \theta) = \frac{n_m^{left}}{n_m} H\big(Q_m^{left}(\theta)\big) + \frac{n_m^{right}}{n_m} H\big(Q_m^{right}(\theta)\big)$$

and CART selects $\theta^* = \arg\min_\theta G(Q_m, \theta)$. It then recurses on both children until a stopping rule is met (maximum depth reached, too few samples, or the node is pure).

### Impurity Measures $H(\cdot)$

**Classification.** Let $p_{mk}$ be the proportion of class $k$ in node $m$.

1. **Gini impurity** — the probability of misclassifying a randomly drawn observation if we label it randomly according to the class distribution of the node:
   $$H(Q_m) = \sum_{k} p_{mk}(1 - p_{mk}) = 1 - \sum_k p_{mk}^2$$

2. **Entropy** (its decrease is called *information gain*; this is what ID3/C4.5 use):
   $$H(Q_m) = -\sum_{k} p_{mk} \log_2 p_{mk}$$

3. **Misclassification error** — $H(Q_m) = 1 - \max_k p_{mk}$. It is *not* used for growing trees, because it is not sensitive enough to changes in the class probabilities (see the plot in Section 2), but it can be used for pruning.

**Regression.** Let $\bar{y}_m$ be the mean of the target in node $m$.

1. **Mean Squared Error** (variance of the node; the leaf predicts the mean):
   $$H(Q_m) = \frac{1}{n_m}\sum_{y \in Q_m}(y - \bar{y}_m)^2$$

2. **Mean Absolute Error** (the leaf predicts the median; more robust to outliers, much slower to train):
   $$H(Q_m) = \frac{1}{n_m}\sum_{y \in Q_m}|y - \text{median}(y)_m|$$

3. **Poisson deviance** — for count targets (e.g. number of insurance claims).

## Key Hyperparameters of a Decision Tree

A fully grown tree memorises the training set (every leaf is pure), so almost all hyperparameters are about **limiting complexity**.

### 1. Maximum depth (`max_depth`)
- The single most important hyperparameter; `None` (the default!) means "grow until all leaves are pure" → overfitting
- Good practice: start from `max_depth=3` and increase while monitoring the validation score

### 2. Minimum samples to split (`min_samples_split`)
- A node with fewer observations is not split any further
- Small value → overfitting; large value → the tree cannot learn (high bias)

### 3. Minimum samples in a leaf (`min_samples_leaf`)
- A split is only valid if both children keep at least this many observations
- Very effective smoother for regression and for **probability estimates** (a leaf with 1 observation can only output probabilities 0 or 1)

### 4. Splitting criterion (`criterion`)
- Classification: `'gini'` (default), `'entropy'`, `'log_loss'`; regression: `'squared_error'` (default), `'absolute_error'`, `'friedman_mse'`, `'poisson'`
- Gini vs entropy rarely matters in practice

### 5. Number of features considered at each split (`max_features`)
- The lower the value, the more randomness → lower variance but higher bias; crucial for Random Forest

### 6. Cost-complexity pruning (`ccp_alpha`)
- *Post*-pruning: grow a large tree $T_0$ first, then find the subtree $T$ minimising
  $$R_\alpha(T) = R(T) + \alpha |\tilde{T}|$$
  where $R(T)$ is the total impurity of the leaves and $|\tilde{T}|$ is the number of leaves. Larger $\alpha$ → smaller tree.

All of them should be selected with **cross-validation**.

## Advantages and Limitations of a Single Tree

### Advantages ✓
1. **Interpretable**: the model can be drawn and explained to a non-technical audience (important for regulated industries, e.g. credit scoring)
2. **No feature scaling needed**: splits depend only on the *ordering* of feature values, so trees are invariant to monotone transformations of the features
3. **Handles non-linearities and interactions automatically**
4. **Built-in feature selection** and feature importance
5. **Fast** to train and to predict

### Limitations ✗
1. **High variance / instability**: a small change in the training data can produce a completely different tree
2. **Overfitting** unless constrained or pruned
3. **Piecewise-constant predictions**: not smooth, and **cannot extrapolate** beyond the range of the training target (crucial for trending financial series!)
4. **Greedy** training does not guarantee a globally optimal tree
5. **Struggles with linear relationships**: approximating a straight line requires many "staircase" splits
6. **Biased with imbalanced classes** (use `class_weight`)
7. In scikit-learn, **categorical features must be numerically encoded** first

## From a Tree to a Forest: Bagging and Random Forest

A single tree is a low-bias, **high-variance** estimator. **Ensemble methods** combine many models into one; **bagging** (**B**ootstrap **agg**regat**ing**, Breiman 1996) targets exactly the variance:

1. Draw $B$ bootstrap samples (sampling $n$ observations **with replacement**) from the training set
2. Train one deep, unpruned tree $\hat{f}_b$ on each bootstrap sample
3. Aggregate: average the predictions (regression), or average the class probabilities / take a majority vote (classification)
   $$\hat{f}_{bag}(x) = \frac{1}{B}\sum_{b=1}^{B}\hat{f}_b(x)$$

### Why does averaging help — and why is bagging not enough?

If the $B$ trees have variance $\sigma^2$ each and pairwise correlation $\rho$, the variance of their average is

$$\text{Var}\big(\hat{f}_{bag}(x)\big) = \rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$$

The second term vanishes as $B \to \infty$, but the first does not: **the correlation between trees limits the benefit of averaging**. Bagged trees are highly correlated, because a few strong features dominate the top splits of every tree.

**Random Forest** (Breiman 2001) adds **feature bagging**: at *each split*, only a random subset of `max_features` features is considered. This decorrelates the trees (lower $\rho$) at the price of a slightly higher bias of each individual tree — overall a better trade-off.

### Out-of-Bag (OOB) Error

Each bootstrap sample leaves out on average $(1 - 1/n)^n \approx e^{-1} \approx 36.8\%$ of the observations. For every training observation $x_i$ we can therefore build a prediction using **only the trees that did not see $x_i$**. The resulting OOB score is an almost-free estimate of the generalisation error — a built-in alternative to cross-validation.

### Extremely Randomized Trees (Extra Trees)

One more layer of randomness: for every candidate feature the threshold is **drawn at random** and the best of these random splits is used (and, by default, there is no bootstrap). Variance drops a bit more, bias rises a bit more, and training is faster because no threshold search is needed.

## Key Hyperparameters of a Random Forest

All tree hyperparameters above, plus:

### 1. Number of trees (`n_estimators`)
- More is better, but with diminishing returns — beyond some point additional trees only cost time. More trees **never cause overfitting**.

### 2. Number of features per split (`max_features`)
- The main tuning knob. Rules of thumb: $\sqrt{p}$ for classification, $p/3$ to $p$ for regression (scikit-learn defaults: `'sqrt'` for the classifier, `1.0` — i.e. plain bagging — for the regressor!)

### 3. Bootstrap (`bootstrap`, `max_samples`)
- Whether each tree sees a bootstrap sample (almost always yes), and how large it is

### 4. Parallelisation (`n_jobs`)
- Trees are independent, so training is embarrassingly parallel: `n_jobs=-1` uses all cores

## Advantages and Limitations of Random Forest

### Advantages ✓
1. Much **lower variance** than a single tree; robust to overfitting
2. Strong performance **out of the box**, little tuning required
3. Inherits the convenience of trees: no scaling, handles non-linearities and interactions, fairly robust to outliers
4. Built-in **OOB** validation and **feature importances**

### Limitations ✗
1. **No longer interpretable** as a set of rules (hundreds of deep trees)
2. **Expensive** in memory and prediction time for large forests
3. Still **cannot extrapolate**
4. Not a good fit for very **high-dimensional sparse** data (e.g. bag-of-words) — linear models do better there
5. Impurity-based feature importances are **biased** towards high-cardinality features (see Section 4.5)

# 2. From-Scratch Implementation

In this section we implement CART from scratch to understand the mechanics: impurity measures, the exhaustive search for the best split, recursive tree building and prediction. We then reuse our tree to build a Random Forest (bootstrap + feature bagging + OOB score).

## 2.1 Impurity Measures

In [ ]:
# Import necessary libraries
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Matplotlib configuration for better plots
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

print("Libraries imported successfully!")

In [ ]:
def gini(y, n_classes):
    """
    Gini impurity of a node.

    Parameters:
    -----------
    y : array of int, shape (n_samples,)
        Class labels encoded as 0, 1, ..., n_classes - 1
    n_classes : int
        Total number of classes

    Returns:
    --------
    float : Gini impurity, 1 - sum_k p_k^2
    """
    p = np.bincount(y, minlength=n_classes) / len(y)
    return 1.0 - np.sum(p**2)


def entropy(y, n_classes):
    """
    Entropy of a node (in bits).

    Returns:
    --------
    float : Entropy, -sum_k p_k log2(p_k), with the convention 0 * log(0) = 0
    """
    p = np.bincount(y, minlength=n_classes) / len(y)
    p = p[p > 0]
    return -np.sum(p * np.log2(p)) + 0.0  # '+ 0.0' turns the floating-point -0.0 of a pure node into 0.0


# Test the impurity functions
pure = np.array([0, 0, 0, 0])
mixed = np.array([0, 0, 1, 1])
skewed = np.array([0, 0, 0, 1])

for name, node in [("pure", pure), ("50/50", mixed), ("75/25", skewed)]:
    print(f"{name:6s} node -> Gini: {gini(node, 2):.4f}, Entropy: {entropy(node, 2):.4f}")

For a binary problem all three impurity measures are functions of a single number $p$ (the share of class 1). Plotting them explains why Gini and entropy are used for growing trees, while the misclassification error is not: the first two are **strictly concave**, so a split that makes the children "purer" is always rewarded, even if the majority class (and hence the misclassification error) does not change.

In [ ]:
p = np.linspace(0.001, 0.999, 500)

gini_curve = 2 * p * (1 - p)
entropy_curve = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))
misclassification_curve = 1 - np.maximum(p, 1 - p)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(p, gini_curve, linewidth=2, label="Gini impurity")
ax.plot(p, entropy_curve / 2, linewidth=2, label="Entropy (scaled by 1/2)")
ax.plot(p, misclassification_curve, linewidth=2, linestyle="--", label="Misclassification error")
ax.set_xlabel("p (proportion of class 1 in the node)", fontsize=12)
ax.set_ylabel("Impurity", fontsize=12)
ax.set_title("Impurity Measures for Binary Classification", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

**A worked example.** A node holds 400 observations of each class, $(400, 400)$. Compare two candidate splits:

- Split A: children $(300, 100)$ and $(100, 300)$
- Split B: children $(200, 400)$ and $(200, 0)$ — the right child is **pure**

Both splits misclassify 200 observations, so the misclassification error cannot tell them apart. Gini and entropy prefer split B.

In [ ]:
def weighted_impurity(children, impurity_func):
    """Weighted impurity of a split given the class counts of the children."""
    n_total = sum(sum(counts) for counts in children)
    total = 0.0
    for counts in children:
        y_child = np.repeat(np.arange(len(counts)), counts)
        if impurity_func == "misclassification":
            h = 1 - max(counts) / sum(counts)
        else:
            h = impurity_func(y_child, len(counts))
        total += sum(counts) / n_total * h
    return total


split_a = [(300, 100), (100, 300)]
split_b = [(200, 400), (200, 0)]

print(f"{'Criterion':<26s}{'Split A':>10s}{'Split B':>10s}")
for name, func in [("Misclassification error", "misclassification"), ("Gini", gini), ("Entropy", entropy)]:
    print(f"{name:<26s}{weighted_impurity(split_a, func):>10.4f}{weighted_impurity(split_b, func):>10.4f}")

## 2.2 CART from Scratch

The implementation below follows the mathematical formulation from Section 1 literally:

1. **`_best_split`** — for every (candidate) feature and every threshold (midpoints between consecutive sorted unique values), compute the weighted impurity of the children $G(Q_m, \theta)$ and keep the minimum.
2. **`_build`** — recursively split the data until a stopping rule is hit; store the prediction in every node.
3. **`predict`** — route every observation from the root to a leaf.

The classifier and the regressor share all of this logic. They differ only in the **impurity** function and in the **value stored in a leaf** (class proportions vs. mean), so we put the common part in a base class.

We also accumulate the **impurity-based feature importance** (*Mean Decrease in Impurity*, MDI): every split contributes $\frac{n_m}{n}\big(H(Q_m) - G(Q_m, \theta^*)\big)$ to the importance of the feature it uses.

> Real implementations (including scikit-learn) are much faster: they sort each feature once and update the class counts incrementally while sweeping the threshold from left to right, which gives $O(p \cdot n \log n)$ per node instead of our $O(p \cdot n^2)$.

In [ ]:
class Node:
    """A single node of the tree. Leaves have feature=None."""

    def __init__(self, value, impurity, n_samples):
        self.value = value  # prediction stored in the node (class proportions or mean)
        self.impurity = impurity
        self.n_samples = n_samples
        self.feature = None  # index of the splitting feature
        self.threshold = None  # go left if x[feature] <= threshold
        self.left = None
        self.right = None

    @property
    def is_leaf(self):
        return self.feature is None


class BaseDecisionTree:
    """
    CART implemented from scratch - logic shared by the classifier and the regressor.
    """

    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, max_features=None, random_state=None):
        """
        Parameters:
        -----------
        max_depth : int or None
            Maximum depth of the tree (None = grow until the leaves are pure)
        min_samples_split : int
            Minimum number of samples required to split a node
        min_samples_leaf : int
            Minimum number of samples required in each child
        max_features : int or None
            Number of randomly chosen features considered at each split (None = all)
        random_state : int or None
            Seed used for the random choice of features
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state

    # --- the two methods that differ between classification and regression ---
    def _impurity(self, y):
        raise NotImplementedError

    def _leaf_value(self, y):
        raise NotImplementedError

    # --- training ---
    def _fit_tree(self, X, y):
        self.n_features_ = X.shape[1]
        self._rng = np.random.default_rng(self.random_state)
        self._importances = np.zeros(self.n_features_)
        self._n_total = len(y)
        self.root_ = self._build(X, y, depth=0)
        total = self._importances.sum()
        self.feature_importances_ = self._importances / total if total > 0 else self._importances
        return self

    def _best_split(self, X, y):
        """Exhaustive search for the split minimising the weighted impurity of the children."""
        n = len(y)
        best_feature, best_threshold, best_score = None, None, np.inf

        # Feature bagging: consider only a random subset of features (used by Random Forest)
        if self.max_features is None:
            features = np.arange(self.n_features_)
        else:
            features = self._rng.choice(self.n_features_, size=self.max_features, replace=False)

        for j in features:
            values = np.unique(X[:, j])  # sorted unique values
            thresholds = (values[:-1] + values[1:]) / 2  # midpoints
            for t in thresholds:
                left = X[:, j] <= t
                n_left = left.sum()
                n_right = n - n_left
                if n_left < self.min_samples_leaf or n_right < self.min_samples_leaf:
                    continue
                score = n_left / n * self._impurity(y[left]) + n_right / n * self._impurity(y[~left])
                if score < best_score:
                    best_feature, best_threshold, best_score = j, t, score

        return best_feature, best_threshold, best_score

    def _build(self, X, y, depth):
        node = Node(value=self._leaf_value(y), impurity=self._impurity(y), n_samples=len(y))

        # Stopping rules
        if (
            (self.max_depth is not None and depth >= self.max_depth)
            or len(y) < self.min_samples_split
            or node.impurity < 1e-12  # pure node
        ):
            return node

        feature, threshold, score = self._best_split(X, y)
        if feature is None:  # no valid split exists
            return node

        # Impurity-based feature importance (MDI)
        self._importances[feature] += len(y) / self._n_total * (node.impurity - score)

        left = X[:, feature] <= threshold
        node.feature, node.threshold = feature, threshold
        node.left = self._build(X[left], y[left], depth + 1)
        node.right = self._build(X[~left], y[~left], depth + 1)
        return node

    # --- prediction ---
    def _predict_value(self, x):
        node = self.root_
        while not node.is_leaf:
            node = node.left if x[node.feature] <= node.threshold else node.right
        return node.value

    # --- utilities ---
    def get_depth(self):
        def depth(node):
            return 0 if node.is_leaf else 1 + max(depth(node.left), depth(node.right))

        return depth(self.root_)

    def get_n_leaves(self):
        def leaves(node):
            return 1 if node.is_leaf else leaves(node.left) + leaves(node.right)

        return leaves(self.root_)

    def print_tree(self, feature_names=None, node=None, indent=""):
        """Print the learned decision rules."""
        node = self.root_ if node is None else node
        if node.is_leaf:
            print(f"{indent}--> value: {np.round(node.value, 3)} (n={node.n_samples})")
            return
        name = f"x[{node.feature}]" if feature_names is None else feature_names[node.feature]
        print(f"{indent}if {name} <= {node.threshold:.3f}:")
        self.print_tree(feature_names, node.left, indent + "    ")
        print(f"{indent}else:  # {name} > {node.threshold:.3f}")
        self.print_tree(feature_names, node.right, indent + "    ")


class DecisionTreeClassifierScratch(BaseDecisionTree):
    """CART classifier: Gini or entropy impurity, leaves store class proportions."""

    def __init__(self, criterion="gini", **kwargs):
        super().__init__(**kwargs)
        self.criterion = criterion

    def _impurity(self, y):
        return gini(y, self.n_classes_) if self.criterion == "gini" else entropy(y, self.n_classes_)

    def _leaf_value(self, y):
        return np.bincount(y, minlength=self.n_classes_) / len(y)

    def fit(self, X, y):
        # Encode the labels as 0, ..., K-1 (np.unique returns the sorted classes and the encoding)
        self.classes_, y_encoded = np.unique(y, return_inverse=True)
        self.n_classes_ = len(self.classes_)
        return self._fit_tree(np.asarray(X, dtype=float), y_encoded)

    def predict_proba(self, X):
        return np.array([self._predict_value(x) for x in np.asarray(X, dtype=float)])

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


class DecisionTreeRegressorScratch(BaseDecisionTree):
    """CART regressor: MSE (variance) impurity, leaves store the mean of the target."""

    def _impurity(self, y):
        return np.var(y)

    def _leaf_value(self, y):
        return np.mean(y)

    def fit(self, X, y):
        return self._fit_tree(np.asarray(X, dtype=float), np.asarray(y, dtype=float))

    def predict(self, X):
        return np.array([self._predict_value(x) for x in np.asarray(X, dtype=float)])

    def score(self, X, y):
        """R^2 score."""
        y = np.asarray(y, dtype=float)
        return 1 - np.sum((y - self.predict(X)) ** 2) / np.sum((y - y.mean()) ** 2)


print("Decision tree classes created successfully!")

### Train and Test the From-Scratch Classifier

We start with the Iris dataset. Note that there is **no feature scaling step** — trees do not need it (we verify this claim in Section 3.5).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Load data
iris = load_iris()
X = iris.data
y = iris.target

print("Iris Dataset Loaded")
print(f"Features shape: {X.shape}")
print(f"Feature names: {iris.feature_names}")
print(f"Target names: {iris.target_names}")

# Split the data into training and testing sets (no scaling needed for trees!)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

In [ ]:
# Create and train the model
tree_scratch = DecisionTreeClassifierScratch(criterion="gini", max_depth=3)

start_time = time.time()
tree_scratch.fit(X_train, y_train)
training_time = time.time() - start_time

print(f"Training time: {training_time:.4f} seconds")
print(f"Depth: {tree_scratch.get_depth()}, number of leaves: {tree_scratch.get_n_leaves()}")
print(f"Training accuracy: {tree_scratch.score(X_train, y_train):.4f}")
print(f"Test accuracy:     {tree_scratch.score(X_test, y_test):.4f}")

print("\nLearned decision rules (leaf value = class proportions):\n")
tree_scratch.print_tree(feature_names=iris.feature_names)

### Sanity Check Against Scikit-learn

With the same hyperparameters, our implementation and `DecisionTreeClassifier` should learn the same tree — **unless two candidate splits have exactly the same impurity**. Such ties are then broken differently: we keep the first split found, while scikit-learn visits the features in a random order (this is why a *deterministic* algorithm has a `random_state`).

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_sklearn = DecisionTreeClassifier(criterion="gini", max_depth=3, random_state=42)
tree_sklearn.fit(X_train, y_train)

agreement = np.mean(tree_scratch.predict(X_test) == tree_sklearn.predict(X_test))

print(f"{'':<22s}{'From scratch':>14s}{'scikit-learn':>14s}")
print(
    f"{'Training accuracy':<22s}{tree_scratch.score(X_train, y_train):>14.4f}{tree_sklearn.score(X_train, y_train):>14.4f}"
)
print(f"{'Test accuracy':<22s}{tree_scratch.score(X_test, y_test):>14.4f}{tree_sklearn.score(X_test, y_test):>14.4f}")
print(f"{'Number of leaves':<22s}{tree_scratch.get_n_leaves():>14d}{tree_sklearn.get_n_leaves():>14d}")
print(f"\nShare of identical test predictions: {agreement:.4f}")

print("\nFeature importances (MDI):")
importance_df = pd.DataFrame(
    {"From scratch": tree_scratch.feature_importances_, "scikit-learn": tree_sklearn.feature_importances_},
    index=iris.feature_names,
)
print(importance_df.round(4))

# Where do the two trees differ? Look at the node reached by: petal length > 2.45 and petal width > 1.55
node_mask = (X_train[:, 2] > 2.45) & (X_train[:, 3] > 1.55)
X_node, y_node = X_train[node_mask], y_train[node_mask]
print(f"\nNode with {node_mask.sum()} observations, class counts: {np.bincount(y_node, minlength=3)}")
for feature, threshold, chosen_by in [(2, 4.85, "from scratch"), (3, 1.70, "scikit-learn")]:
    left = X_node[:, feature] <= threshold
    score = left.mean() * gini(y_node[left], 3) + (~left).mean() * gini(y_node[~left], 3)
    print(
        f"  {iris.feature_names[feature]} <= {threshold:.2f} -> weighted Gini = {score:.6f}  (chosen by: {chosen_by})"
    )

**What to notice:** the two trees are identical down to one node, where two different splits achieve *exactly* the same weighted Gini. Both trees are equally good on the training data, yet they disagree on a couple of test observations. This is a first glimpse of the **instability** of decision trees, to which we return in Section 3.4.

### Visualize Decision Boundaries (2D)

Because every split compares **one feature** with a threshold, the decision regions of a tree are always **axis-aligned rectangles**. The deeper the tree, the finer (and the more overfitted) the partition.

In [ ]:
from matplotlib.colors import ListedColormap

# Use only 2 features for visualization: petal length and petal width
X_2d = X[:, [2, 3]]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(X_2d, y, test_size=0.3, random_state=42, stratify=y)

cmap_light = ListedColormap(["#FFAAAA", "#AAFFAA", "#AAAAFF"])
cmap_bold = ["#FF0000", "#00AA00", "#0000FF"]

# Create mesh grid
h = 0.02
x_min, x_max = X_2d[:, 0].min() - 0.5, X_2d[:, 0].max() + 0.5
y_min, y_max = X_2d[:, 1].min() - 0.5, X_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, depth in zip(axes, [1, 3, None]):
    # We use scikit-learn here only because predicting ~100k mesh points with our Python loop is slow
    model = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train_2d, y_train_2d)
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.8)
    for i, color in enumerate(cmap_bold):
        idx = y_train_2d == i
        ax.scatter(X_train_2d[idx, 0], X_train_2d[idx, 1], c=color, label=iris.target_names[i], edgecolor="black", s=50)
    ax.set_xlabel(iris.feature_names[2], fontsize=12)
    ax.set_ylabel(iris.feature_names[3], fontsize=12)
    ax.set_title(
        f"max_depth={depth} | train acc: {model.score(X_train_2d, y_train_2d):.3f}, "
        f"test acc: {model.score(X_test_2d, y_test_2d):.3f}",
        fontsize=13,
        fontweight="bold",
    )
    ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

### The From-Scratch Regression Tree

The same code with the variance as the impurity measure gives a regression tree. On a one-dimensional problem we can *see* the piecewise-constant nature of the model: each leaf is a horizontal segment at the mean of its observations.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Noisy sine wave
rng = np.random.default_rng(42)
X_sin = np.sort(rng.uniform(0, 6, size=120)).reshape(-1, 1)
y_sin = np.sin(X_sin).ravel() + rng.normal(0, 0.25, size=120)
X_plot = np.linspace(0, 6, 600).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))

for ax, depth in zip(axes, [2, 4, None]):
    reg_scratch = DecisionTreeRegressorScratch(max_depth=depth).fit(X_sin, y_sin)
    reg_sklearn = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_sin, y_sin)

    ax.scatter(X_sin, y_sin, s=20, color="gray", alpha=0.6, label="Training data")
    ax.plot(X_plot, np.sin(X_plot), color="black", linestyle=":", linewidth=2, label="True function")
    ax.plot(X_plot, reg_scratch.predict(X_plot), color="red", linewidth=2.5, label="From scratch")
    ax.plot(X_plot, reg_sklearn.predict(X_plot), color="blue", linewidth=1.2, linestyle="--", label="scikit-learn")
    ax.set_title(
        f"max_depth={depth} | leaves: {reg_scratch.get_n_leaves()} | train R²: {reg_scratch.score(X_sin, y_sin):.3f}",
        fontsize=13,
        fontweight="bold",
    )
    ax.set_xlabel("x", fontsize=12)
    ax.set_ylabel("y", fontsize=12)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("max_depth=2    -> underfitting: 4 steps cannot follow the sine wave")
print("max_depth=4    -> reasonable approximation")
print("max_depth=None -> overfitting: one leaf per observation, the tree reproduces the noise (train R² = 1)")

## 2.3 Random Forest from Scratch

With a working tree, a Random Forest needs only three more ingredients:

1. **Bootstrap**: every tree is trained on $n$ observations sampled *with replacement*
2. **Feature bagging**: every tree uses `max_features` random features at each split (already supported by our tree)
3. **Aggregation**: average the class probabilities of all trees (this *soft voting* is also what scikit-learn does)

For free we get the **out-of-bag score**: every observation is predicted only by the trees that did not see it during training.

In [ ]:
class RandomForestClassifierScratch:
    """
    Random Forest classifier built on top of DecisionTreeClassifierScratch.
    """

    def __init__(self, n_estimators=25, max_features="sqrt", max_depth=None, min_samples_leaf=1, random_state=None):
        """
        Parameters:
        -----------
        n_estimators : int
            Number of trees in the forest
        max_features : 'sqrt', int or None
            Number of features considered at each split (None = all features, i.e. plain bagging)
        max_depth, min_samples_leaf :
            Passed to every tree
        random_state : int or None
            Seed for the bootstrap samples and the feature bagging
        """
        self.n_estimators = n_estimators
        self.max_features = max_features
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.random_state = random_state

    def fit(self, X, y):
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        n, p = X.shape
        rng = np.random.default_rng(self.random_state)
        self.classes_ = np.unique(y)
        max_features = max(1, int(np.sqrt(p))) if self.max_features == "sqrt" else self.max_features

        self.trees_ = []
        oob_proba = np.zeros((n, len(self.classes_)))
        oob_counts = np.zeros(n)

        for _ in range(self.n_estimators):
            # 1. Bootstrap sample: n indices drawn with replacement
            idx = rng.integers(0, n, size=n)
            oob_mask = np.ones(n, dtype=bool)
            oob_mask[idx] = False  # observations NOT drawn are out-of-bag for this tree

            # 2. Tree with feature bagging
            tree = DecisionTreeClassifierScratch(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf,
                max_features=max_features,
                random_state=int(rng.integers(0, 2**31 - 1)),
            )
            tree.fit(X[idx], y[idx])
            self.trees_.append(tree)

            # 3. Accumulate the OOB predictions of this tree
            oob_proba[oob_mask] += self._aligned_proba(tree, X[oob_mask])
            oob_counts[oob_mask] += 1

        # OOB score: accuracy on observations that were out-of-bag at least once
        has_oob = oob_counts > 0
        oob_pred = self.classes_[np.argmax(oob_proba[has_oob], axis=1)]
        self.oob_score_ = np.mean(oob_pred == y[has_oob])

        self.feature_importances_ = np.mean([t.feature_importances_ for t in self.trees_], axis=0)
        return self

    def _aligned_proba(self, tree, X):
        """A bootstrap sample may miss a class entirely - align the tree's columns with self.classes_."""
        proba = np.zeros((len(X), len(self.classes_)))
        proba[:, np.searchsorted(self.classes_, tree.classes_)] = tree.predict_proba(X)
        return proba

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        return np.mean([self._aligned_proba(tree, X) for tree in self.trees_], axis=0)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    def score(self, X, y):
        return np.mean(self.predict(X) == y)


print("RandomForestClassifierScratch class created successfully!")

### Single Tree vs. Forest on a Harder Dataset

Iris is too easy to show the benefit of an ensemble, so we switch to the Breast Cancer Wisconsin dataset (569 observations, 30 features, binary target). Our pure-Python implementation is slow, so this cell takes several seconds.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score

cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.3, random_state=42, stratify=y_c)

print(f"Breast Cancer dataset: {X_c.shape[0]} samples, {X_c.shape[1]} features")
print(f"Class distribution: {dict(zip(cancer.target_names, np.bincount(y_c)))}")

# Single fully grown tree (from scratch)
start_time = time.time()
single_tree = DecisionTreeClassifierScratch().fit(X_train_c, y_train_c)
tree_time = time.time() - start_time

# Random forest (from scratch)
start_time = time.time()
forest_scratch = RandomForestClassifierScratch(n_estimators=100, max_features="sqrt", random_state=42)
forest_scratch.fit(X_train_c, y_train_c)
forest_time = time.time() - start_time

print("\n" + "=" * 60)
print("FROM-SCRATCH MODELS ON BREAST CANCER")
print("=" * 60)
print(f"{'Model':<28s}{'Train acc':>10s}{'Test acc':>10s}{'Test AUC':>10s}{'Time [s]':>10s}")
for name, model, elapsed in [
    ("Single tree (full depth)", single_tree, tree_time),
    ("Random forest (100 trees)", forest_scratch, forest_time),
]:
    test_auc = roc_auc_score(y_test_c, model.predict_proba(X_test_c)[:, 1])
    print(
        f"{name:<28s}{model.score(X_train_c, y_train_c):>10.4f}{model.score(X_test_c, y_test_c):>10.4f}"
        f"{test_auc:>10.4f}{elapsed:>10.2f}"
    )
print(f"\nOOB accuracy of the forest: {forest_scratch.oob_score_:.4f}  <- computed without touching the test set!")

**What to notice:**
- Both models reach 100% training accuracy — for tree-based models the training score says nothing about quality.
- The gain in test **accuracy** is modest. Keep in mind that the test set has only 171 observations, so a single observation is worth 0.6 percentage points: differences of this size are within the noise of the split.
- The gain in test **ROC AUC** is large. A fully grown tree can only output probabilities of 0 or 1, so it ranks observations very crudely, whereas the forest averages 100 such votes into a fine-grained score (more on this in Section 4.6).
- The **OOB accuracy** is in the same range as the test accuracy, although it was computed from the training data only.
- Each tree in the forest evaluates only $\lfloor\sqrt{30}\rfloor = 5$ features per split, so the forest of 100 trees is far less than 100 times slower than the single tree.

# 3. Decision Trees with Scikit-learn

Now that we understand how CART works internally, let's use scikit-learn's optimised implementation. In this section we:

1. inspect a fitted tree (`plot_tree`, `export_text`),
2. watch a tree overfit as it gets deeper,
3. control its complexity with cost-complexity pruning,
4. demonstrate the instability of trees (the motivation for Random Forest),
5. verify that trees do not need feature scaling,
6. look at regression trees and their inability to extrapolate.

## 3.1 Inspecting a Fitted Tree

Interpretability is the main selling point of a single tree. `plot_tree` draws it and `export_text` prints the rules. Each node shows the split, the impurity, the number of samples and the class distribution (`value`).

In [ ]:
from sklearn.tree import export_text, plot_tree

tree_clf = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_clf.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(
    tree_clf,
    feature_names=iris.feature_names,
    class_names=list(iris.target_names),
    filled=True,  # colour = majority class, intensity = purity
    rounded=True,
    impurity=True,
    fontsize=11,
    ax=ax,
)
plt.title("Decision Tree on Iris (max_depth=3)", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

print("The same tree as text rules:\n")
print(export_text(tree_clf, feature_names=iris.feature_names))

## 3.2 Overfitting: Depth vs. Performance

We grow trees of increasing depth on the Breast Cancer data and compare the **training** score with the **cross-validated** score (computed on the training set only — the test set stays untouched until the very end, see Section 5).

`validation_curve` does exactly this: for every value of one hyperparameter it runs a full cross-validation and returns the training and validation scores of every fold.

In [ ]:
from sklearn.model_selection import StratifiedKFold, validation_curve

depths = np.arange(1, 16)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

train_scores, valid_scores = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X_train_c,
    y_train_c,
    param_name="max_depth",
    param_range=depths,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

train_mean, train_std = train_scores.mean(axis=1), train_scores.std(axis=1)
valid_mean, valid_std = valid_scores.mean(axis=1), valid_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(depths, train_mean, "o-", label="Training accuracy", linewidth=2, markersize=7)
ax.fill_between(depths, train_mean - train_std, train_mean + train_std, alpha=0.15)
ax.plot(depths, valid_mean, "s-", label="Cross-validation accuracy", linewidth=2, markersize=7)
ax.fill_between(depths, valid_mean - valid_std, valid_mean + valid_std, alpha=0.15)
ax.set_xlabel("max_depth", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Decision Tree: Accuracy vs. Depth (5-fold CV on the training set)", fontsize=14, fontweight="bold")
ax.set_xticks(depths)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

best_depth = depths[np.argmax(valid_mean)]
print(f"Best max_depth according to CV: {best_depth} (CV accuracy: {valid_mean.max():.4f})")
print(
    f"Depth of the unconstrained tree: {DecisionTreeClassifier(random_state=42).fit(X_train_c, y_train_c).get_depth()}"
)
print("\nThe training accuracy climbs to 1.0, while the CV accuracy peaks early and then stagnates or drops.")
print("The growing gap between the two curves is the signature of overfitting.")

## 3.3 Cost-Complexity Pruning

Limiting `max_depth` is **pre-pruning**: we stop early, and we stop *every branch* at the same depth. **Post-pruning** is more flexible: grow the full tree and then remove the branches that do not pay for their complexity, by minimising

$$R_\alpha(T) = R(T) + \alpha|\tilde{T}|$$

`cost_complexity_pruning_path` returns the *effective alphas* — the values of $\alpha$ at which the optimal subtree changes. We then pick $\alpha$ by cross-validation, like any other hyperparameter.

In [ ]:
from sklearn.model_selection import cross_val_score

# 1. Compute the pruning path on the training data
full_tree = DecisionTreeClassifier(random_state=42)
path = full_tree.cost_complexity_pruning_path(X_train_c, y_train_c)
ccp_alphas = path.ccp_alphas[:-1]  # the last alpha prunes the tree down to the root - skip it

print(f"Number of candidate alphas: {len(ccp_alphas)}")

# 2. Cross-validate a tree for every alpha
n_leaves, cv_means, cv_stds = [], [], []
for alpha in ccp_alphas:
    pruned = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    scores_alpha = cross_val_score(pruned, X_train_c, y_train_c, cv=cv, scoring="accuracy")
    cv_means.append(scores_alpha.mean())
    cv_stds.append(scores_alpha.std())
    n_leaves.append(pruned.fit(X_train_c, y_train_c).get_n_leaves())

cv_means, cv_stds = np.array(cv_means), np.array(cv_stds)
best_alpha = ccp_alphas[np.argmax(cv_means)]

# 3. Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(ccp_alphas, n_leaves, "o-", drawstyle="steps-post", linewidth=2)
axes[0].set_xlabel("ccp_alpha", fontsize=12)
axes[0].set_ylabel("Number of leaves", fontsize=12)
axes[0].set_title("Tree Size vs. alpha", fontsize=14, fontweight="bold")

axes[1].errorbar(ccp_alphas, cv_means, yerr=cv_stds, fmt="s-", linewidth=2, capsize=3)
axes[1].axvline(best_alpha, color="red", linestyle="--", label=f"best alpha = {best_alpha:.4f}")
axes[1].set_xlabel("ccp_alpha", fontsize=12)
axes[1].set_ylabel("CV accuracy (mean ± std over folds)", fontsize=12)
axes[1].set_title("Cross-Validated Accuracy vs. alpha", fontsize=14, fontweight="bold")
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

pruned_tree = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha).fit(X_train_c, y_train_c)
full_tree.fit(X_train_c, y_train_c)
print(f"Unpruned tree: {full_tree.get_n_leaves():3d} leaves, depth {full_tree.get_depth()}")
print(
    f"Pruned tree:   {pruned_tree.get_n_leaves():3d} leaves, depth {pruned_tree.get_depth()} (CV accuracy {cv_means.max():.4f})"
)

## 3.4 Instability: The Achilles' Heel of Trees

A tree is a **high-variance** estimator: because the splits are hierarchical, a small change at the top propagates to everything below. Let's train four trees on four bootstrap samples of the *same* training set and look at their first splits.

In [ ]:
rng = np.random.default_rng(0)

print("=" * 70)
print("TREES TRAINED ON DIFFERENT BOOTSTRAP SAMPLES OF THE SAME DATA")
print("=" * 70)

bootstrap_predictions = []
for b in range(4):
    idx = rng.integers(0, len(X_train_c), size=len(X_train_c))
    tree_b = DecisionTreeClassifier(random_state=42).fit(X_train_c[idx], y_train_c[idx])
    bootstrap_predictions.append(tree_b.predict(X_test_c))

    root_feature = cancer.feature_names[tree_b.tree_.feature[0]]
    root_threshold = tree_b.tree_.threshold[0]
    print(
        f"Tree {b + 1}: root split = '{root_feature}' <= {root_threshold:.3f} | "
        f"leaves: {tree_b.get_n_leaves():2d} | test acc: {tree_b.score(X_test_c, y_test_c):.4f}"
    )

bootstrap_predictions = np.array(bootstrap_predictions)
disagreement = np.mean(bootstrap_predictions.min(axis=0) != bootstrap_predictions.max(axis=0))
print(f"\nShare of test observations on which the four trees do NOT all agree: {disagreement:.2%}")
print("\nEach tree tells a different 'story' about the data - a problem for interpretability,")
print("but exactly the diversity that bagging needs: averaging unstable models reduces variance.")

## 3.5 Trees Do Not Need Feature Scaling

KNN and SVM rely on distances, so they break down without scaling. A tree only asks "is $x_j \leq t$?", which depends on the **order** of the values, not on their magnitude. Any strictly increasing transformation of a feature (standardisation, $\log$, ...) moves the thresholds but leaves the partition of the observations — and hence the predictions — unchanged.

In [ ]:
from sklearn.preprocessing import StandardScaler

# All Breast Cancer features are non-negative, so log1p is a valid monotone transformation
variants = {
    "Raw features": (X_train_c, X_test_c),
    "Standardised (z-score)": tuple(StandardScaler().fit(X_train_c).transform(part) for part in (X_train_c, X_test_c)),
    "log(1 + x)": (np.log1p(X_train_c), np.log1p(X_test_c)),
}

reference_pred = None
print(f"{'Transformation':<26s}{'Test accuracy':>14s}{'Same predictions as raw?':>28s}")
for name, (X_tr, X_te) in variants.items():
    pred = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_train_c).predict(X_te)
    reference_pred = pred if reference_pred is None else reference_pred
    print(f"{name:<26s}{np.mean(pred == y_test_c):>14.4f}{str(np.array_equal(pred, reference_pred)):>28s}")

print("\nPractical consequence: pipelines for tree-based models need no scaler.")
print(
    "(Rarely, thresholds falling between two very close values can flip a prediction due to floating-point rounding.)"
)

## 3.6 Regression Trees

We now predict California house prices with `DecisionTreeRegressor`. As before, the hyperparameter is selected with cross-validation on the training set.

> **Note on subsampling.** To keep the notebook fast we work with 5,000 of the 20,640 rows. We draw them **without replacement** — sampling with replacement would put exact copies of some rows on both sides of the train/test split, which is a form of data leakage.

In [ ]:
from sklearn.datasets import fetch_california_housing

# Load California Housing dataset and subsample WITHOUT replacement
housing = fetch_california_housing()
subsample_idx = np.random.default_rng(42).choice(len(housing.target), size=5000, replace=False)
X_h, y_h = housing.data[subsample_idx], housing.target[subsample_idx]

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

print("California Housing Dataset (subsample)")
print(f"Training set: {X_train_h.shape}, test set: {X_test_h.shape}")
print(f"Features: {housing.feature_names}")
print("Target: median house value (in $100,000s)")

# Validation curve over min_samples_leaf - a very effective smoother for regression trees
leaf_sizes = [1, 2, 5, 10, 20, 35, 50, 100, 200]
train_scores_h, valid_scores_h = validation_curve(
    DecisionTreeRegressor(random_state=42),
    X_train_h,
    y_train_h,
    param_name="min_samples_leaf",
    param_range=leaf_sizes,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(leaf_sizes, -train_scores_h.mean(axis=1), "o-", label="Training RMSE", linewidth=2, markersize=7)
ax.plot(leaf_sizes, -valid_scores_h.mean(axis=1), "s-", label="Cross-validation RMSE", linewidth=2, markersize=7)
ax.set_xscale("log")
ax.set_xlabel("min_samples_leaf (log scale)", fontsize=12)
ax.set_ylabel("RMSE", fontsize=12)
ax.set_title("Regression Tree: RMSE vs. Minimum Leaf Size", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

best_leaf = leaf_sizes[int(np.argmin(-valid_scores_h.mean(axis=1)))]
print(f"Best min_samples_leaf according to CV: {best_leaf}")
print("min_samples_leaf=1 -> training RMSE of 0 (memorisation) but a poor CV RMSE: classic overfitting.")

### Trees Cannot Extrapolate

A leaf predicts the mean of the training targets that fell into it, so the prediction of a tree (and of any average of trees, i.e. a forest) can **never leave the range of target values seen in training**. For cross-sectional problems this is harmless. For anything that trends — prices, index levels, GDP, assets under management — it is fatal.

The standard remedy in finance is to model a **stationary** transformation of the target (returns instead of prices, growth rates instead of levels) and to avoid raw time or level variables as features.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

# A trending "price-like" series: linear trend + cycle + noise
rng = np.random.default_rng(42)
t = np.arange(200).reshape(-1, 1)
series = 100 + 0.5 * t.ravel() + 5 * np.sin(t.ravel() / 8) + rng.normal(0, 2, size=200)

# Train on the first 150 periods, "forecast" the last 50
t_train, t_test = t[:150], t[150:]
s_train, s_test = series[:150], series[150:]

models_extrapolation = {
    "Linear regression": LinearRegression(),
    "Decision tree": DecisionTreeRegressor(min_samples_leaf=5, random_state=42),
    "Random forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=42),
}

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(t_train, s_train, color="gray", linewidth=1.5, label="Training data")
ax.plot(t_test, s_test, color="black", linewidth=1.5, linestyle=":", label="Future (unseen) data")
for name, model in models_extrapolation.items():
    model.fit(t_train, s_train)
    ax.plot(t_test, model.predict(t_test), linewidth=2.5, label=f"{name} forecast")
ax.axvline(150, color="red", linestyle="--", alpha=0.6)
ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Level", fontsize=12)
ax.set_title("Tree-Based Models Cannot Extrapolate a Trend", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Maximum target value seen in training: {s_train.max():.2f}")
for name, model in models_extrapolation.items():
    print(f"{name:<18s} max forecast: {model.predict(t_test).max():7.2f}")

# 4. Random Forest with Scikit-learn

## 4.1 Single Tree vs. Bagging vs. Random Forest vs. Extra Trees

We compare four models with the same cross-validation folds on the Breast Cancer training set. `BaggingClassifier` with trees is a Random Forest *without* feature bagging (every split sees all 30 features), which lets us isolate the effect of decorrelating the trees.

We report two metrics: **accuracy** (based on hard class predictions) and **ROC AUC** (based on predicted probabilities — the metric of the classification task in our Kaggle League).

In [ ]:
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.model_selection import cross_validate

models_comparison = {
    "Single tree (unpruned)": DecisionTreeClassifier(random_state=42),
    "Bagging (200 trees)": BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, random_state=42, n_jobs=-1),
    "Random forest (200 trees)": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Extra trees (200 trees)": ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1),
}

rows = []
for name, model in models_comparison.items():
    start_time = time.time()
    res = cross_validate(model, X_train_c, y_train_c, cv=cv, scoring=["accuracy", "roc_auc"])
    rows.append(
        {
            "Model": name,
            "CV accuracy": res["test_accuracy"].mean(),
            "std": res["test_accuracy"].std(),
            "CV ROC AUC": res["test_roc_auc"].mean(),
            "std ": res["test_roc_auc"].std(),
            "Time [s]": time.time() - start_time,
        }
    )

print("=" * 90)
print("5-FOLD CROSS-VALIDATION ON THE TRAINING SET")
print("=" * 90)
print(pd.DataFrame(rows).round(4).to_string(index=False))

print("\nThe ensembles beat the single tree, and the gap is largest for ROC AUC: a fully grown tree")
print("outputs only probabilities of 0 or 1, so it ranks observations very coarsely (see Section 4.6).")

### Decision Boundaries: Tree vs. Forest

On a two-dimensional toy dataset we can see what averaging does: the jagged, axis-aligned boundary of a single deep tree becomes much smoother, and the predicted probabilities (colour intensity) become graded instead of 0/1.

In [ ]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=400, noise=0.3, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_moons, y_moons, test_size=0.3, random_state=42)

xx_m, yy_m = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.5, 2, 300))
grid_m = np.c_[xx_m.ravel(), yy_m.ravel()]

models_moons = {
    "Single tree (unpruned)": DecisionTreeClassifier(random_state=42),
    "Single tree (min_samples_leaf=10)": DecisionTreeClassifier(min_samples_leaf=10, random_state=42),
    "Random forest (300 trees)": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
}

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, (name, model) in zip(axes, models_moons.items()):
    model.fit(X_train_m, y_train_m)
    proba = model.predict_proba(grid_m)[:, 1].reshape(xx_m.shape)
    ax.contourf(xx_m, yy_m, proba, levels=20, cmap="RdBu", alpha=0.8)
    ax.scatter(X_train_m[:, 0], X_train_m[:, 1], c=y_train_m, cmap="RdBu", edgecolor="black", s=30)
    ax.set_title(
        f"{name}\ntrain acc: {model.score(X_train_m, y_train_m):.3f}, test acc: {model.score(X_test_m, y_test_m):.3f}",
        fontsize=13,
        fontweight="bold",
    )
    ax.set_xlabel("Feature 1", fontsize=12)
    ax.set_ylabel("Feature 2", fontsize=12)

plt.tight_layout()
plt.show()

## 4.2 Number of Trees and the OOB Error

Two facts to verify empirically:

1. **More trees never hurt** the generalisation error — the OOB error decreases and then flattens out. `n_estimators` is therefore not tuned like a usual hyperparameter: use as many trees as you can afford.
2. **`max_features` matters**: it controls the correlation between the trees.

We use `warm_start=True`, which *adds* trees to an already fitted forest instead of retraining from scratch.

In [ ]:
ensemble_settings = {
    "max_features='sqrt' (default, 5 of 30)": "sqrt",
    "max_features=0.33 (10 of 30)": 0.33,
    "max_features=None (30 of 30 = bagging)": None,
}
n_trees_range = list(range(20, 401, 20))

fig, ax = plt.subplots(figsize=(12, 6))

for label, max_features in ensemble_settings.items():
    forest = RandomForestClassifier(
        warm_start=True, oob_score=True, max_features=max_features, random_state=42, n_jobs=-1
    )
    oob_errors = []
    for n_trees in n_trees_range:
        forest.set_params(n_estimators=n_trees)
        forest.fit(X_train_c, y_train_c)
        oob_errors.append(1 - forest.oob_score_)
    ax.plot(n_trees_range, oob_errors, linewidth=2, label=label)

ax.set_xlabel("n_estimators", fontsize=12)
ax.set_ylabel("OOB error rate", fontsize=12)
ax.set_title("Random Forest: OOB Error vs. Number of Trees", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("The curves are noisy for small forests and flatten out after roughly 100-200 trees.")
print("On this dataset the ranking is usually sqrt < 0.33 < all features (bagging), but the differences are small")
print("(about half a percentage point) and depend on the random seed - always verify with your own data.")

### How Good Is the OOB Estimate?

The OOB score is obtained from a **single fit**. Let's compare it with 5-fold cross-validation (five fits) and with the test set.

In [ ]:
forest_oob = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=42, n_jobs=-1)

start_time = time.time()
forest_oob.fit(X_train_c, y_train_c)
oob_time = time.time() - start_time

start_time = time.time()
cv_acc = cross_val_score(
    RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1), X_train_c, y_train_c, cv=cv
).mean()
cv_time = time.time() - start_time

print("=" * 60)
print("THREE ESTIMATES OF THE GENERALISATION ACCURACY")
print("=" * 60)
print(f"OOB accuracy (1 fit):        {forest_oob.oob_score_:.4f}   [{oob_time:.2f} s]")
print(f"5-fold CV accuracy (5 fits): {cv_acc:.4f}   [{cv_time:.2f} s]")
print(f"Test accuracy:               {forest_oob.score(X_test_c, y_test_c):.4f}")
print(f"Training accuracy:           {forest_oob.score(X_train_c, y_train_c):.4f}   <- useless as a quality measure")

## 4.3 Random Forest Regression

Back to California Housing. Remember that scikit-learn's `RandomForestRegressor` uses `max_features=1.0` by default — i.e. it is plain **bagging**. Setting `max_features` to around one third of the features (the recommendation of the original paper) is often better, so it is worth tuning.

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

models_regression = {
    "Single tree (unpruned)": DecisionTreeRegressor(random_state=42),
    f"Single tree (min_samples_leaf={best_leaf})": DecisionTreeRegressor(min_samples_leaf=best_leaf, random_state=42),
    "RF, max_features=1.0 (bagging)": RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
    "RF, max_features=0.33": RandomForestRegressor(n_estimators=300, max_features=0.33, random_state=42, n_jobs=-1),
    "Extra trees, max_features=0.33": ExtraTreesRegressor(
        n_estimators=300, max_features=0.33, random_state=42, n_jobs=-1
    ),
}

rows = []
for name, model in models_regression.items():
    start_time = time.time()
    res = cross_validate(
        model, X_train_h, y_train_h, cv=5, scoring=["neg_root_mean_squared_error", "r2"], return_train_score=True
    )
    rows.append(
        {
            "Model": name,
            "Train RMSE": -res["train_neg_root_mean_squared_error"].mean(),
            "CV RMSE": -res["test_neg_root_mean_squared_error"].mean(),
            "CV R²": res["test_r2"].mean(),
            "Time [s]": time.time() - start_time,
        }
    )

print("=" * 90)
print("REGRESSION: 5-FOLD CROSS-VALIDATION ON THE TRAINING SET")
print("=" * 90)
print(pd.DataFrame(rows).round(4).to_string(index=False))

## 4.4 Looking Inside the Forest

A fitted forest exposes its trees in `estimators_`. Each tree is an ordinary (deep!) `DecisionTree*` object, so we can examine how different the individual trees are, and see variance reduction at work: **individual trees are mediocre, their average is good**.

In [ ]:
forest_reg = RandomForestRegressor(n_estimators=300, max_features=0.33, random_state=42, n_jobs=-1)
forest_reg.fit(X_train_h, y_train_h)

# Predictions of every individual tree on the test set: shape (n_trees, n_test)
tree_predictions = np.array([tree.predict(X_test_h) for tree in forest_reg.estimators_])
individual_rmse = np.sqrt(((tree_predictions - y_test_h) ** 2).mean(axis=1))

# RMSE of the average of the first B trees
cumulative_mean = np.cumsum(tree_predictions, axis=0) / np.arange(1, len(tree_predictions) + 1).reshape(-1, 1)
cumulative_rmse = np.sqrt(((cumulative_mean - y_test_h) ** 2).mean(axis=1))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(individual_rmse, bins=30, alpha=0.7, edgecolor="black", label="Individual trees")
axes[0].axvline(cumulative_rmse[-1], color="red", linewidth=2.5, label=f"Forest: {cumulative_rmse[-1]:.3f}")
axes[0].set_xlabel("Test RMSE", fontsize=12)
axes[0].set_ylabel("Number of trees", fontsize=12)
axes[0].set_title("Each Tree Alone vs. the Forest", fontsize=14, fontweight="bold")
axes[0].legend(fontsize=11)

axes[1].plot(np.arange(1, len(cumulative_rmse) + 1), cumulative_rmse, linewidth=2)
axes[1].set_xscale("log")
axes[1].set_xlabel("Number of trees averaged (log scale)", fontsize=12)
axes[1].set_ylabel("Test RMSE", fontsize=12)
axes[1].set_title("Variance Reduction by Averaging", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

depths_in_forest = [tree.get_depth() for tree in forest_reg.estimators_]
print(
    f"Depth of the trees in the forest: min {min(depths_in_forest)}, max {max(depths_in_forest)} (unpruned by design)"
)
print(f"Mean RMSE of an individual tree: {individual_rmse.mean():.4f}")
print(f"RMSE of the forest:              {cumulative_rmse[-1]:.4f}")

# Average pairwise correlation between the errors of the trees - the rho from the variance formula
errors = tree_predictions[:100] - y_test_h
rho = np.corrcoef(errors)[np.triu_indices(100, k=1)].mean()
print(f"\nAverage pairwise correlation between the errors of the trees (first 100 trees): {rho:.3f}")
print("The lower this correlation, the more the forest gains from averaging - this is what max_features controls.")

## 4.5 Feature Importance: MDI vs. Permutation Importance

Random Forest offers a built-in importance measure, `feature_importances_`: the **Mean Decrease in Impurity (MDI)** — the same quantity we accumulated in our from-scratch tree, averaged over all trees. It is fast and convenient, but it has two well-documented flaws:

1. It is computed on the **training data**, so it rewards features the forest has merely *overfitted* to.
2. It is **biased towards high-cardinality features** (continuous variables, IDs): more unique values mean more candidate thresholds and more chances to find a spuriously good split.

**Permutation importance** avoids both: shuffle one feature in a **held-out** set, and measure how much the score drops. If the model did not really need the feature, nothing happens.

To expose the problem we use a dataset with a weaker signal than Breast Cancer — the *German Credit* data (1,000 loan applicants, good/bad credit risk; we study it in detail in Section 6) — and add two features of **pure noise**: one continuous (high cardinality) and one binary (low cardinality). A trustworthy importance measure must give both an importance of (approximately) zero.

In [ ]:
from pathlib import Path

from sklearn.inspection import permutation_importance
from sklearn.preprocessing import OrdinalEncoder

# German Credit data (shipped with the repository); categorical features are integer-encoded for the forest
credit = pd.read_csv(Path("..") / "data" / "german_credit.csv")
X_imp = credit.drop(columns="credit_risk")
y_imp = (credit["credit_risk"] == "bad").astype(int)
categorical_columns = X_imp.select_dtypes(exclude="number").columns
X_imp[categorical_columns] = OrdinalEncoder().fit_transform(X_imp[categorical_columns])

# Add two pure-noise features
rng = np.random.default_rng(42)
X_imp = X_imp.assign(
    NOISE_continuous=rng.normal(size=len(X_imp)),
    NOISE_binary=rng.integers(0, 2, size=len(X_imp)),
)

X_train_i, X_test_i, y_train_i, y_test_i = train_test_split(
    X_imp, y_imp, test_size=0.25, random_state=42, stratify=y_imp
)
forest_imp = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(X_train_i, y_train_i)

# MDI importance (training data, built-in)
mdi = pd.Series(forest_imp.feature_importances_, index=X_imp.columns)

# Permutation importance (test data, drop in ROC AUC, 20 shuffles per feature)
perm = permutation_importance(
    forest_imp, X_test_i, y_test_i, scoring="roc_auc", n_repeats=20, random_state=42, n_jobs=-1
)
perm_mean = pd.Series(perm.importances_mean, index=X_imp.columns)
perm_std = pd.Series(perm.importances_std, index=X_imp.columns)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
for ax, values, errors_bar, title in [
    (axes[0], mdi, None, "MDI Importance (training data)"),
    (axes[1], perm_mean, perm_std, "Permutation Importance (test data, drop in ROC AUC)"),
]:
    order = values.sort_values().index
    colors = ["red" if "NOISE" in name else "steelblue" for name in order]
    ax.barh(order, values[order], xerr=None if errors_bar is None else errors_bar[order], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

rank_mdi = mdi.rank(ascending=False).astype(int)
print(f"{'':<20s}{'MDI':>10s}{'MDI rank':>10s}{'Permutation importance (mean ± std)':>40s}")
for name in ["NOISE_continuous", "NOISE_binary"]:
    perm_text = f"{perm_mean[name]:+.4f} ± {perm_std[name]:.4f}"
    print(f"{name:<20s}{mdi[name]:>10.4f}{rank_mdi[name]:>7d}/{len(mdi)}{perm_text:>40s}")

top_feature = perm_mean.idxmax()
print(f"\nFor scale - the most important feature by permutation: '{top_feature}' ({perm_mean[top_feature]:+.4f})")

**What to notice:**
- MDI ranks the *continuous* noise feature among the **most important** variables of the model, next to genuinely predictive ones such as the credit amount or the checking account status. The *binary* noise feature, in contrast, sits near the bottom. Both are equally useless — the only difference is the number of distinct values, i.e. the number of split points the trees can try. This is the cardinality bias in action, amplified by the fact that MDI is measured on the training data, where deep trees happily "use" noise to separate the last few observations.
- The permutation importance of both noise features is **tiny** — a change in ROC AUC of well under 0.01, an order of magnitude below the most important feature. It is not exactly zero, and it may even come out slightly positive or negative: the reported standard deviation only reflects the randomness of the shuffling, not the sampling noise of a 250-client test set. Treat values of this size as zero.
- **A caveat for permutation importance:** when features are strongly correlated, shuffling one of them leaves the information available through the others, so *each* of them looks unimportant. Low permutation importance therefore means "the model can do without this feature **given the others**", not "this feature is unrelated to the target". Possible remedies: cluster the correlated features and keep one per cluster, or permute whole groups together.

## 4.6 Predicted Probabilities

For a tree, `predict_proba` returns the class proportions of the training observations in the leaf. In a fully grown tree every leaf is pure, so the "probabilities" are only 0 or 1 — useless for ranking clients by risk, and heavily punished by the log-loss. A forest averages many such trees and produces much finer-grained scores. Increasing `min_samples_leaf` makes the scores of both models finer still — whether that also improves the metric is an empirical question that cross-validation has to answer.

In [ ]:
from sklearn.metrics import log_loss, roc_auc_score

models_proba = {
    "Tree (unpruned)": DecisionTreeClassifier(random_state=42),
    "Tree (min_samples_leaf=10)": DecisionTreeClassifier(min_samples_leaf=10, random_state=42),
    "Random forest (300 trees)": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "Random forest (300 trees, min_samples_leaf=3)": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
}

fig, axes = plt.subplots(1, 4, figsize=(22, 4.5), sharey=True)
print(f"{'Model':<48s}{'Unique proba':>13s}{'Test AUC':>10s}{'Test log-loss':>15s}")
for ax, (name, model) in zip(axes, models_proba.items()):
    proba_test = model.fit(X_train_c, y_train_c).predict_proba(X_test_c)[:, 1]
    ax.hist(proba_test, bins=25, range=(0, 1), edgecolor="black", alpha=0.75)
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted P(y = 1)", fontsize=11)
    print(
        f"{name:<48s}{len(np.unique(proba_test)):>13d}{roc_auc_score(y_test_c, proba_test):>10.4f}"
        f"{log_loss(y_test_c, proba_test):>15.4f}"
    )
axes[0].set_ylabel("Number of test observations", fontsize=11)
plt.tight_layout()
plt.show()

print(
    "\nFor the single tree, min_samples_leaf improves AUC and log-loss dramatically; for the forest the effect is small"
)
print("and can go either way. Tip for the Kaggle League (ROC AUC): always submit predict_proba, never predict.")

# 5. Hyperparameter Tuning with Cross-Validation

Cross-validation itself was covered in detail in the previous labs, so here we go straight to tuning. Two rules apply throughout:

- **The search only ever sees the training set.** The test set is used once, at the very end.
- `best_score_` of a search is the *maximum* over many noisy CV estimates, so it is an **optimistically biased** estimate of the generalisation performance. Report the test-set score (or use nested cross-validation) instead.

## 5.1 GridSearchCV for a Decision Tree

A single tree is cheap, so an exhaustive grid is affordable.

In [ ]:
from sklearn.model_selection import GridSearchCV

print("=" * 60)
print("GRIDSEARCHCV - DECISION TREE")
print("=" * 60)

param_grid_tree = {
    "max_depth": [2, 3, 4, 5, 7, 10, None],
    "min_samples_leaf": [1, 3, 5, 10, 20],
    "criterion": ["gini", "entropy"],
}

grid_search_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=param_grid_tree,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True,
)
grid_search_tree.fit(X_train_c, y_train_c)

print(f"\nCombinations tested: {len(grid_search_tree.cv_results_['params'])}")
print(f"Best parameters: {grid_search_tree.best_params_}")
print(f"Best CV ROC AUC: {grid_search_tree.best_score_:.4f}")

# Visualize: heatmap of CV AUC for max_depth x min_samples_leaf (criterion = best)
cv_results_tree = pd.DataFrame(grid_search_tree.cv_results_)
best_criterion = grid_search_tree.best_params_["criterion"]
subset = cv_results_tree[cv_results_tree["param_criterion"] == best_criterion].copy()
subset["param_max_depth"] = subset["param_max_depth"].astype(object).where(subset["param_max_depth"].notna(), "None")
pivot = subset.pivot(index="param_max_depth", columns="param_min_samples_leaf", values="mean_test_score")
pivot = pivot.loc[[d if d is not None else "None" for d in param_grid_tree["max_depth"]]]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="YlGnBu", ax=ax, cbar_kws={"label": "CV ROC AUC"})
ax.set_title(f"Decision Tree Grid Search (criterion='{best_criterion}')", fontsize=14, fontweight="bold")
ax.set_xlabel("min_samples_leaf", fontsize=12)
ax.set_ylabel("max_depth", fontsize=12)
plt.tight_layout()
plt.show()

## 5.2 RandomizedSearchCV for a Random Forest

A forest is two orders of magnitude more expensive than a tree and has more hyperparameters, so we sample the space randomly. Practical advice:

- **Do not tune `n_estimators`** — fix it at a value that is "large enough" (and increase it for the final model).
- Focus on **`max_features`** and **`min_samples_leaf`** (or `max_depth`); `max_samples` (size of the bootstrap sample) is a useful extra regulariser.

In [ ]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

print("=" * 60)
print("RANDOMIZEDSEARCHCV - RANDOM FOREST")
print("=" * 60)

param_distributions_rf = {
    "max_features": uniform(0.05, 0.75),  # fraction of features considered per split, from [0.05, 0.80]
    "min_samples_leaf": randint(1, 21),
    "max_depth": [None, 4, 6, 8, 12],
    "max_samples": [None, 0.5, 0.75],
    "criterion": ["gini", "entropy"],
}

random_search_rf = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=200, random_state=42),
    param_distributions=param_distributions_rf,
    n_iter=30,
    cv=cv,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
)

start_time = time.time()
random_search_rf.fit(X_train_c, y_train_c)
print(f"\nSearch time: {time.time() - start_time:.1f} s for 30 candidates x 5 folds = 150 forests")

print(f"Best parameters: {random_search_rf.best_params_}")
print(f"Best CV ROC AUC: {random_search_rf.best_score_:.4f}")

cv_results_rf = pd.DataFrame(random_search_rf.cv_results_).sort_values("rank_test_score")
print("\n" + "=" * 60)
print("TOP 5 CONFIGURATIONS")
print("=" * 60)
top_columns = [c for c in cv_results_rf.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score"]
print(cv_results_rf[top_columns].head().round(4).to_string(index=False))

### Final Check on the Test Set

Only now — with all modelling decisions made — do we touch the test set, **once**.

In [ ]:
default_forest = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_train_c, y_train_c)

final_models = {
    "Tuned decision tree": grid_search_tree.best_estimator_,
    "Random forest (defaults)": default_forest,
    "Tuned random forest": random_search_rf.best_estimator_,
}

print("=" * 70)
print("TEST SET PERFORMANCE")
print("=" * 70)
print(f"{'Model':<28s}{'CV AUC (train)':>16s}{'Test AUC':>10s}{'Test accuracy':>15s}")
cv_auc_default = cross_val_score(default_forest, X_train_c, y_train_c, cv=cv, scoring="roc_auc").mean()
cv_aucs = [grid_search_tree.best_score_, cv_auc_default, random_search_rf.best_score_]
for (name, model), cv_auc in zip(final_models.items(), cv_aucs):
    test_auc = roc_auc_score(y_test_c, model.predict_proba(X_test_c)[:, 1])
    print(f"{name:<28s}{cv_auc:>16.4f}{test_auc:>10.4f}{model.score(X_test_c, y_test_c):>15.4f}")

print("\nNote how little tuning adds to a Random Forest with default settings -")
print("one of the reasons why it is such a popular baseline.")

# 6. Comprehensive Example: Credit Scoring Pipeline

Let's put everything together on a finance problem: predicting whether a bank client is a **bad credit risk**. We use the classic *German Credit* dataset (UCI Statlog; shipped with this repository as `data/german_credit.csv`; 1,000 loan applicants, 20 features — 7 numerical and 13 categorical — and a good/bad label).

This example adds three things that real-world data demands:

- **Categorical features.** scikit-learn trees need numbers. For tree-based models an `OrdinalEncoder` is usually sufficient (and keeps the dimensionality low): a tree can isolate any category with a couple of splits, so the arbitrary order of the integer codes does far less harm than it would in a linear model. One-hot encoding is the alternative.
- **Class imbalance** (70% good / 30% bad) → `class_weight="balanced_subsample"` and a probability-based metric (ROC AUC).
- **Asymmetric costs.** The dataset comes with a cost matrix: granting a loan to a bad client costs **5**, rejecting a good client costs **1**. The default 0.5 probability cut-off ignores this, so we choose the cut-off that minimises the expected cost — using **out-of-fold predictions on the training set**, never the test set.

## Complete Workflow:
1. Load and explore data
2. Split into train/test sets
3. Build a pipeline with preprocessing
4. Hyperparameter tuning with cross-validation
5. Choose the decision threshold on out-of-fold predictions
6. Evaluate the final model on the test set
7. Interpret the model

In [ ]:
print("=" * 70)
print("COMPLETE CREDIT SCORING PIPELINE - BEST PRACTICES")
print("=" * 70)

# Step 1: Load data (shipped with the repository - see data/README.md for the source and licence)
print("\n[STEP 1] Loading Data...")
credit = pd.read_csv(Path("..") / "data" / "german_credit.csv")
X_credit = credit.drop(columns="credit_risk")
y_credit = (credit["credit_risk"] == "bad").astype(int)  # positive class (1) = bad credit risk

categorical_features = X_credit.select_dtypes(exclude="number").columns.tolist()
numerical_features = X_credit.select_dtypes(include="number").columns.tolist()

print(f"Dataset shape: {X_credit.shape}")
print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Share of bad clients: {y_credit.mean():.1%}")
print(f"Missing values: {int(X_credit.isna().sum().sum())}")

X_credit.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Step 2: Train/Test Split (IMPORTANT: Split BEFORE any preprocessing!)
print("[STEP 2] Splitting Data...")
X_train_cr, X_test_cr, y_train_cr, y_test_cr = train_test_split(
    X_credit, y_credit, test_size=0.25, random_state=42, stratify=y_credit
)
print(f"Training set: {X_train_cr.shape[0]} samples, test set: {X_test_cr.shape[0]} samples")

# Step 3: Build Pipeline
print("\n[STEP 3] Building Pipeline...")
preprocessor = ColumnTransformer(
    transformers=[
        # categories never seen during fit are encoded as -1 instead of raising an error
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_features),
    ],
    remainder="passthrough",  # numerical features are passed through untouched - no scaling for trees!
    verbose_feature_names_out=False,
)

credit_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("rf", RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample", random_state=42)),
    ]
)
print("Pipeline created: OrdinalEncoder (categorical) + passthrough (numerical) → Random Forest")

# Step 4: Hyperparameter Tuning with Cross-Validation
print("\n[STEP 4] Hyperparameter Tuning...")
print("Using RandomizedSearchCV with stratified 5-fold cross-validation, scoring = ROC AUC")

param_distributions_credit = {
    "rf__max_features": uniform(0.1, 0.6),
    "rf__min_samples_leaf": randint(1, 31),
    "rf__max_depth": [None, 4, 6, 8, 12],
    "rf__max_samples": [None, 0.5, 0.75],
}

credit_search = RandomizedSearchCV(
    credit_pipeline,
    param_distributions=param_distributions_credit,
    n_iter=30,
    cv=cv,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
)

# Fit on TRAINING data only
credit_search.fit(X_train_cr, y_train_cr)

print(f"Best parameters found: {credit_search.best_params_}")
print(f"Best cross-validation ROC AUC: {credit_search.best_score_:.4f}")

# Baseline for comparison: a single tuned tree in the same pipeline
tree_pipeline = Pipeline(
    [("preprocessor", preprocessor), ("tree", DecisionTreeClassifier(class_weight="balanced", random_state=42))]
)
tree_search = GridSearchCV(
    tree_pipeline,
    param_grid={"tree__max_depth": [2, 3, 4, 5, 6, 8], "tree__min_samples_leaf": [5, 10, 20, 40]},
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
).fit(X_train_cr, y_train_cr)
print(f"\nBaseline - tuned single tree, CV ROC AUC: {tree_search.best_score_:.4f} ({tree_search.best_params_})")

In [ ]:
from sklearn.model_selection import cross_val_predict

# Step 5: Choose the decision threshold on OUT-OF-FOLD predictions of the training set
print("[STEP 5] Choosing the Decision Threshold...")

COST_FN = 5  # bad client predicted as good -> the loan is granted and defaults
COST_FP = 1  # good client predicted as bad -> lost business

best_pipeline = credit_search.best_estimator_
oof_proba = cross_val_predict(best_pipeline, X_train_cr, y_train_cr, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]


def average_cost(y_true, proba, threshold):
    """Average misclassification cost per client for a given probability cut-off."""
    y_true = np.asarray(y_true)
    pred = (proba >= threshold).astype(int)
    fn = np.sum((y_true == 1) & (pred == 0))
    fp = np.sum((y_true == 0) & (pred == 1))
    return (COST_FN * fn + COST_FP * fp) / len(y_true)


thresholds = np.linspace(0.05, 0.95, 91)
oof_costs = np.array([average_cost(y_train_cr, oof_proba, thr) for thr in thresholds])
best_threshold = thresholds[np.argmin(oof_costs)]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresholds, oof_costs, linewidth=2)
ax.axvline(best_threshold, color="red", linestyle="--", label=f"Cost-minimising cut-off = {best_threshold:.2f}")
ax.axvline(0.5, color="gray", linestyle=":", label="Default cut-off = 0.50")
ax.set_xlabel("Probability cut-off for predicting 'bad'", fontsize=12)
ax.set_ylabel("Average cost per client (out-of-fold)", fontsize=12)
ax.set_title("Choosing the Decision Threshold under Asymmetric Costs (FN = 5, FP = 1)", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Theory: with calibrated probabilities, predict 'bad' when p >= FP / (FP + FN).
# class_weight='balanced' multiplies the odds of 'bad' by n_good / n_bad, so the cut-off has to be moved accordingly.
theoretical_threshold = COST_FP / (COST_FP + COST_FN)
odds_adjusted = (COST_FP / COST_FN) * (np.sum(y_train_cr == 0) / np.sum(y_train_cr == 1))
adjusted_threshold = odds_adjusted / (1 + odds_adjusted)

print(f"Theoretical cut-off for calibrated probabilities, FP / (FP + FN): {theoretical_threshold:.3f}")
print(f"Theoretical cut-off after accounting for the class weights:       {adjusted_threshold:.3f}")
print(f"Empirical optimum on out-of-fold predictions:                     {best_threshold:.3f}")
print("\nThe cost curve is flat and noisy around its minimum (750 clients only), so the empirical optimum")
print("is itself an estimate - any cut-off in the flat region is a defensible choice.")

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

# Step 6: Evaluate on Test Set
print("[STEP 6] Final Model Evaluation on Test Set...")
test_proba = best_pipeline.predict_proba(X_test_cr)[:, 1]
test_proba_tree = tree_search.best_estimator_.predict_proba(X_test_cr)[:, 1]

print("\n" + "=" * 70)
print("FINAL MODEL PERFORMANCE")
print("=" * 70)
print(f"Test ROC AUC - Random Forest: {roc_auc_score(y_test_cr, test_proba):.4f}")
print(f"Test ROC AUC - single tree:   {roc_auc_score(y_test_cr, test_proba_tree):.4f}")
print(f"Test PR AUC (average precision) - Random Forest: {average_precision_score(y_test_cr, test_proba):.4f}")
print(f"  (baseline = share of bad clients = {y_test_cr.mean():.4f})")

print(f"\n{'Decision rule':<40s}{'Average cost per client':>25s}")
print(f"{'Grant everyone a loan':<40s}{average_cost(y_test_cr, test_proba, 1.01):>25.4f}")
print(f"{'Reject everyone':<40s}{average_cost(y_test_cr, test_proba, 0.0):>25.4f}")
print(f"{'Random Forest, cut-off 0.50':<40s}{average_cost(y_test_cr, test_proba, 0.5):>25.4f}")
print(
    f"{f'Random Forest, cut-off {best_threshold:.2f} (chosen on OOF)':<40s}"
    f"{average_cost(y_test_cr, test_proba, best_threshold):>25.4f}"
)

y_pred_cr = (test_proba >= best_threshold).astype(int)
print("\n" + "=" * 70)
print(f"DETAILED CLASSIFICATION REPORT (cut-off = {best_threshold:.2f})")
print("=" * 70)
print(classification_report(y_test_cr, y_pred_cr, target_names=["good", "bad"]))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
RocCurveDisplay.from_predictions(y_test_cr, test_proba, name="Random Forest", ax=axes[0])
RocCurveDisplay.from_predictions(y_test_cr, test_proba_tree, name="Single tree", ax=axes[0])
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[0].set_title("ROC Curve - Test Set", fontsize=14, fontweight="bold")

cm_credit = confusion_matrix(y_test_cr, y_pred_cr)
ConfusionMatrixDisplay(cm_credit, display_labels=["good", "bad"]).plot(cmap="Greens", ax=axes[1], values_format="d")
axes[1].set_title(f"Confusion Matrix (cut-off = {best_threshold:.2f})", fontsize=14, fontweight="bold")
axes[1].grid(False)
plt.tight_layout()
plt.show()

In [ ]:
# Step 7: Interpret the model - permutation importance on the test set
print("[STEP 7] Model Interpretation...")

# Computed on the whole pipeline, so the importances refer to the ORIGINAL columns
perm_credit = permutation_importance(
    best_pipeline, X_test_cr, y_test_cr, scoring="roc_auc", n_repeats=30, random_state=42, n_jobs=-1
)
perm_credit_df = pd.DataFrame(
    {"importance": perm_credit.importances_mean, "std": perm_credit.importances_std}, index=X_test_cr.columns
).sort_values("importance")

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(perm_credit_df.index, perm_credit_df["importance"], xerr=perm_credit_df["std"], color="steelblue")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Drop in test ROC AUC when the feature is shuffled", fontsize=12)
ax.set_title("Permutation Importance - Credit Scoring Model", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Most important features:")
print(perm_credit_df.sort_values("importance", ascending=False).head(5).round(4))
print("\nThe test set has only 250 clients, so the importances are noisy (see the error bars):")
print("small negative values are noise, not evidence that a feature 'hurts'.")

# 7. Key Takeaways and Best Practices

## Summary of Decision Trees

**Strengths:**
- ✅ Interpretable: can be drawn, printed as rules and explained to a regulator
- ✅ No feature scaling, no distributional assumptions
- ✅ Captures non-linearities and interactions automatically
- ✅ Fast to train and to predict
- ✅ Built-in feature selection

**Weaknesses:**
- ❌ High variance: unstable and prone to overfitting
- ❌ Piecewise-constant predictions; cannot extrapolate
- ❌ Coarse probability estimates
- ❌ Greedy training — no guarantee of a globally optimal tree
- ❌ Usually clearly less accurate than ensembles

## Summary of Random Forest

**Strengths:**
- ✅ Much lower variance than a single tree (bagging + feature bagging)
- ✅ Excellent out-of-the-box performance, little tuning needed
- ✅ Free validation through the OOB score
- ✅ Trivially parallel (`n_jobs=-1`)
- ✅ Feature importances (prefer permutation importance)

**Weaknesses:**
- ❌ Not interpretable as a set of rules
- ❌ Large memory footprint and slower predictions
- ❌ Still cannot extrapolate
- ❌ Weak on high-dimensional sparse data
- ❌ On tabular data usually (slightly) beaten by gradient boosting — the topic of *Machine Learning in Finance II*

## Best Practices

### 1. **Never Deploy an Unconstrained Single Tree**
```python
# The defaults (max_depth=None, min_samples_leaf=1) memorise the training set
tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20)   # pre-pruning
tree = DecisionTreeClassifier(ccp_alpha=0.005)                    # post-pruning
```

### 2. **Skip the Scaler, Keep the Pipeline**
```python
# Trees do not need scaling, but encoders and imputers still belong inside the pipeline
pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([('cat', OrdinalEncoder(...), categorical_features)],
                                       remainder='passthrough')),
    ('rf', RandomForestClassifier())
])
```

### 3. **Do Not Tune `n_estimators`**
- Fix it at a few hundred; more trees never overfit, they only cost time
- Check the OOB error curve to see where it flattens

### 4. **Tune What Matters**
- Tree: `max_depth`, `min_samples_leaf`, `ccp_alpha`
- Forest: `max_features` first, then `min_samples_leaf` / `max_depth`, optionally `max_samples`
- Remember that `RandomForestRegressor` defaults to `max_features=1.0` (bagging) — try ~0.33

### 5. **Use the OOB Score for Quick Iterations**
- `oob_score=True` gives a validation estimate from a single fit
- Still use proper cross-validation for the final comparison of different model families

### 6. **Prefer Permutation Importance to MDI**
- MDI is biased towards high-cardinality features and is computed on the training data
- Compute permutation importance on held-out data; beware of correlated features

### 7. **Handle Imbalanced Data**
- `class_weight='balanced'` (tree) or `'balanced_subsample'` (forest)
- Evaluate with ROC AUC / PR AUC rather than accuracy, and choose the threshold according to the business costs

### 8. **Care About Probabilities**
- Submit `predict_proba` when the metric is ROC AUC or log-loss
- Increase `min_samples_leaf` for smoother probabilities; consider calibration (`CalibratedClassifierCV`) when the probabilities themselves are used, e.g. as PDs in credit risk

### 9. **Mind Extrapolation in Financial Data**
- Model stationary targets (returns, growth rates), not trending levels
- For time series use `TimeSeriesSplit`, never shuffled K-fold — and note that the OOB score is also invalid there, because bootstrap sampling ignores the time order

### 10. **Cross-Validation Strategy**
- Never tune hyperparameters on the test set!
- `best_score_` of a search is optimistically biased — report the test score or use nested cross-validation
- Stratify splits for classification to maintain class balance

## When to Use What

✅ **Use a single Decision Tree when:**
- Interpretability is a hard requirement (simple scorecards, policy rules, regulatory constraints)
- You want a quick look at the main interactions in a new dataset
- You need to explain the building block of forests and boosting

✅ **Use a Random Forest when:**
- You need a strong baseline for tabular data with minimal preprocessing and tuning
- The data contains mixed feature types, non-linearities, interactions and outliers
- You have enough memory and do not need to extrapolate

❌ **Avoid tree-based models when:**
- The relationship is essentially linear and the data is scarce (a linear model is more efficient)
- The data is very high-dimensional and sparse (text)
- Predictions must extrapolate beyond the training range
- Smooth predictions are required

## Common Pitfalls to Avoid

1. **Leaving `max_depth=None` on a single tree** → memorised training set
2. **Judging a forest by its training accuracy** → it is always close to 100%
3. **Trusting MDI importances blindly** → noise features can look important
4. **Reading causal stories into a single tree** → a different sample gives a different tree
5. **Grid-searching `n_estimators`** → wasted computation
6. **One-hot encoding high-cardinality categoricals for trees** → many sparse, weak features dilute the splits
7. **Using `predict` instead of `predict_proba` for an AUC-scored task** → large loss of ranking information
8. **Forecasting trending series in levels** → flat-line predictions

## Further Reading

- [Scikit-learn Decision Trees Documentation](https://scikit-learn.org/stable/modules/tree.html)
- [Scikit-learn Ensembles: Random Forests and Extra Trees](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- [Permutation Importance vs. Random Forest Feature Importance (MDI)](https://scikit-learn.org/stable/auto_examples/inspection/plot_permutation_importance.html)
- [MLU-Explain: Decision Trees](https://mlu-explain.github.io/decision-tree/) and [Random Forest](https://mlu-explain.github.io/random-forest/)
- James, G., Witten, D., Hastie, T., Tibshirani, R. (2021). *An Introduction to Statistical Learning*, Chapter 8
- Hastie, T., Tibshirani, R., Friedman, J. (2009). *The Elements of Statistical Learning*, Chapters 9.2 and 15
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5–32

# 8. Homework Assignment

## Predicting Credit Card Default with Decision Trees and Random Forest

### Objective
Apply everything you've learned about tree-based models to predict whether a credit card client will default on next month's payment.

### Dataset
Use the **Default of Credit Card Clients** dataset (UCI Machine Learning Repository; 30,000 clients of a Taiwanese bank, 23 features: credit limit, demographic variables, six months of repayment status, bill amounts and payment amounts). It ships with the repository:

```python
import pandas as pd

df = pd.read_csv("../data/credit_card_default.csv.gz")
X, y = df.drop(columns="default_next_month"), df["default_next_month"]  # y = 1 -> default next month
```

- Binary classification: default (≈ 22%) vs. no default
- Main metric: **ROC AUC**
- The meaning of the columns is described on the [UCI page](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients) and in [`data/README.md`](../data/README.md)

### Tasks

#### Part 1: Data Exploration and Preprocessing
1. Load the dataset and give the columns meaningful names (e.g. `PAY_0` is the repayment status in September 2005)
2. Explore the dataset:
   - Print shape, data types and class distribution
   - Create visualizations (histograms, default rate by repayment status, correlation matrix)
   - Check for missing values and undocumented category codes
3. Split data into train (70%) and test (30%) sets with stratification
4. Explain which preprocessing steps a tree-based model needs here, and which it does **not** need

#### Part 2: From-Scratch Implementation
1. Use the from-scratch `DecisionTreeClassifierScratch` (Section 2.2) on a random subsample of 2,000 training observations with `max_depth=3`
2. Verify that scikit-learn finds the same root split
3. Extend the class with **one** of the following and test it:
   - the `min_impurity_decrease` stopping rule, or
   - a faster `_best_split` that sorts each feature once and updates the class counts incrementally (compare the running times)

#### Part 3: Scikit-learn Decision Tree
1. Train an unconstrained tree and report training vs. cross-validated ROC AUC — what do you observe?
2. Plot validation curves for `max_depth` and `min_samples_leaf`
3. Apply cost-complexity pruning and select `ccp_alpha` by cross-validation
4. Visualize the final tree with `plot_tree` (restrict the plot to depth 3) and interpret the first splits in business terms

#### Part 4: Random Forest
1. Train a Random Forest with default settings and `oob_score=True`; compare the OOB score with 5-fold CV
2. Plot the OOB error against `n_estimators` for three values of `max_features`
3. Compare single tree vs. bagging vs. Random Forest vs. Extra Trees (CV ROC AUC and training time)
4. Compare MDI and permutation importance; add a random noise feature and comment on its ranking

#### Part 5: Hyperparameter Tuning
1. Use `GridSearchCV` to tune the decision tree:
   - max_depth: [3, 4, 5, 6, 8, 10]
   - min_samples_leaf: [1, 10, 25, 50, 100]
   - criterion: ['gini', 'entropy']
2. Use `RandomizedSearchCV` (at least 25 iterations) to tune the Random Forest:
   - max_features: uniform(0.1, 0.7)
   - min_samples_leaf: randint(1, 51)
   - max_depth: [None, 6, 10, 14]
   - max_samples: [None, 0.5, 0.75]
3. Compare the two searches: best scores, number of evaluations, computational time
4. Visualize the search results (heatmaps or plots)

#### Part 6: Final Model and Analysis
1. Evaluate the tuned tree and the tuned forest on the test set (ROC AUC, PR AUC, confusion matrix)
2. Assume that a missed default costs 5 times more than a false alarm: choose the probability cut-off on out-of-fold predictions and report the average cost on the test set
3. Create a comprehensive performance report including:
   - All evaluation metrics
   - ROC curves of all models on one plot
   - Comparison table of all approaches (including your best models from the previous labs: logistic regression, KNN, SVM)
4. **Bonus**: Discuss:
   - Which features drive the default risk, and is this consistent with your economic intuition?
   - How well calibrated are the forest's probabilities (`CalibrationDisplay`)? Does `CalibratedClassifierCV` help?
   - The data is a snapshot from 2005 — what would happen to the model after a change of the economic regime (model drift)?
   - Would you be allowed to use `sex`, `age` and `marriage` in a real credit scoring model in the EU?